# FoodGuard AI - Data Preparation

This notebook:
1. **E-Nose Data**
2. **E-Tongue Data**
3. **Vision Deposit Patterns**
4. **Demo Samples**

---

## 1. Imports & Configuration

In [ ]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import json
from PIL import Image, ImageDraw
import random
from tqdm import tqdm

sys.path.insert(0, '..')

from foodguard_lib import (
    DB_PATH, ADULTERANTS,
    insert_aroma_analysis, insert_taste_analysis, insert_vision_analysis,
    execute_insert, generate_batch_id, generate_vision_images
)

print("[OK] Imports successful")
print(f"Adulterants: {ADULTERANTS}")

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

## 2. E-Nose Synthetic Data Generation

**Features**: ammonia, alcohol, voc, sulfur, hydrocarbon (5 sensors)

**Range Strategy**: Each adulterant has characteristic sensor signatures (from literature).

In [ ]:
def generate_enose_data(n_samples_per_class=1000):
    """
    Generate E-Nose synthetic data.
    Returns DataFrame with columns: adulterant, ammonia, alcohol, voc, sulfur, hydrocarbon
    """

    # Sensor ranges and shifts per adulterant (from literature)
    enose_profiles = {
        "Fresh": {
            "ammonia": (0.1, 0.5),
            "alcohol": (0.2, 0.8),
            "voc": (0.3, 0.9),
            "sulfur": (0.05, 0.3),
            "hydrocarbon": (0.1, 0.6)
        },
        "Water": {  # Diluted signals
            "ammonia": (0.05, 0.25),
            "alcohol": (0.1, 0.4),
            "voc": (0.1, 0.5),
            "sulfur": (0.02, 0.15),
            "hydrocarbon": (0.05, 0.3)
        },
        "Urea": {  # High ammonia
            "ammonia": (1.5, 3.0),
            "alcohol": (0.2, 0.8),
            "voc": (0.3, 0.9),
            "sulfur": (0.1, 0.5),
            "hydrocarbon": (0.1, 0.6)
        },
        "Detergent": {  # Variable profile, some alcohol
            "ammonia": (0.2, 0.6),
            "alcohol": (0.5, 1.5),
            "voc": (0.8, 2.0),
            "sulfur": (0.2, 0.8),
            "hydrocarbon": (0.2, 1.0)
        },
        "Formalin": {  # Pungent signals
            "ammonia": (0.5, 1.5),
            "alcohol": (1.5, 3.5),
            "voc": (2.0, 4.5),
            "sulfur": (0.5, 1.5),
            "hydrocarbon": (0.3, 1.0)
        },
        "H2O2": {  # Oxidative markers
            "ammonia": (0.1, 0.4),
            "alcohol": (0.3, 1.0),
            "voc": (0.5, 1.5),
            "sulfur": (0.05, 0.25),
            "hydrocarbon": (0.05, 0.3)
        },
        "Spoiled": {  # Acidic fermentation markers
            "ammonia": (0.3, 1.0),
            "alcohol": (0.5, 2.0),
            "voc": (0.8, 2.5),
            "sulfur": (0.3, 1.0),
            "hydrocarbon": (0.1, 0.8)
        }
    }

    data = []
    for adulterant in ADULTERANTS:
        profile = enose_profiles[adulterant]
        for _ in range(n_samples_per_class):
            sample = {"adulterant": adulterant}
            for sensor, (min_val, max_val) in profile.items():
                # Add Gaussian noise around the range
                base = np.random.uniform(min_val, max_val)
                noise = np.random.normal(0, 0.1 * (max_val - min_val))
                sample[sensor] = max(0, base + noise)
            data.append(sample)

    return pd.DataFrame(data)

print("[OK] E-Nose generation function defined")

In [ ]:
# Generate E-Nose data
print("Generating E-Nose synthetic data...")
enose_df = generate_enose_data(n_samples_per_class=1000)
print(f"[OK] Generated {len(enose_df)} E-Nose samples")
print(f"\nE-Nose data sample:")
print(enose_df.head(10))
print(f"\nE-Nose by class:")
print(enose_df["adulterant"].value_counts())

## 3. E-Tongue Synthetic Data Generation

**Features**: sweetness, saltiness, sourness, bitterness, umami, astringency (6 sensors)

In [ ]:
def generate_etongue_data(n_samples_per_class=1000):
    """
    Generate E-Tongue synthetic data.
    Returns DataFrame with columns: adulterant, sweetness, saltiness, sourness, bitterness, umami, astringency
    """

    etongue_profiles = {
        "Fresh": {
            "sweetness": (4.0, 6.0),
            "saltiness": (1.0, 2.5),
            "sourness": (0.5, 1.5),
            "bitterness": (0.1, 0.5),
            "umami": (0.5, 1.5),
            "astringency": (0.2, 0.8)
        },
        "Water": {  # Diluted taste
            "sweetness": (2.0, 3.5),
            "saltiness": (0.3, 1.0),
            "sourness": (0.2, 0.8),
            "bitterness": (0.05, 0.2),
            "umami": (0.1, 0.5),
            "astringency": (0.05, 0.3)
        },
        "Urea": {  # High bitterness
            "sweetness": (2.0, 4.0),
            "saltiness": (1.5, 3.0),
            "sourness": (0.5, 1.5),
            "bitterness": (2.0, 4.0),  # ** MARKER
            "umami": (0.2, 1.0),
            "astringency": (1.0, 2.0)  # ** MARKER
        },
        "Detergent": {  # Harsh, bitter, soapy
            "sweetness": (0.5, 1.5),
            "saltiness": (2.0, 3.5),
            "sourness": (0.3, 1.0),
            "bitterness": (2.5, 4.5),  # ** MARKER
            "umami": (0.1, 0.5),
            "astringency": (2.0, 3.5)  # ** MARKER
        },
        "Formalin": {  # Pungent, sour, harsh
            "sweetness": (0.2, 1.0),
            "saltiness": (1.0, 2.5),
            "sourness": (3.0, 5.0),  # ** MARKER
            "bitterness": (1.5, 3.0),
            "umami": (0.1, 0.5),
            "astringency": (3.0, 4.5)  # ** MARKER
        },
        "H2O2": {  # Clean, faint
            "sweetness": (3.0, 5.0),
            "saltiness": (0.2, 1.0),
            "sourness": (0.1, 0.5),
            "bitterness": (0.05, 0.3),
            "umami": (0.1, 0.5),
            "astringency": (0.1, 0.5)
        },
        "Spoiled": {  # Sour, acidic
            "sweetness": (1.0, 2.5),
            "saltiness": (1.5, 2.5),
            "sourness": (3.5, 5.5),  # ** MARKER
            "bitterness": (0.5, 1.5),
            "umami": (0.1, 0.5),
            "astringency": (1.5, 3.0)  # ** MARKER
        }
    }

    data = []
    for adulterant in ADULTERANTS:
        profile = etongue_profiles[adulterant]
        for _ in range(n_samples_per_class):
            sample = {"adulterant": adulterant}
            for sensor, (min_val, max_val) in profile.items():
                base = np.random.uniform(min_val, max_val)
                noise = np.random.normal(0, 0.15 * (max_val - min_val))
                sample[sensor] = max(0, base + noise)
            data.append(sample)

    return pd.DataFrame(data)

print("[OK] E-Tongue generation function defined")

In [ ]:
# Generate E-Tongue data
print("Generating E-Tongue synthetic data...")
etongue_df = generate_etongue_data(n_samples_per_class=1000)
print(f"[OK] Generated {len(etongue_df)} E-Tongue samples")
print(f"\nE-Tongue data sample:")
print(etongue_df.head(10))
print(f"\nE-Tongue by class:")
print(etongue_df["adulterant"].value_counts())

In [ ]:
# Generate vision images
print("Generating vision deposit pattern images...")
vision_results = generate_vision_images(n_samples_per_class=60)
print(f"[OK] Generated {len(vision_results)} vision images")
print(f"\nSample vision results (first 5):")
for img_path, deposit_type, adulterant in vision_results[:5]:
    print(f"  {adulterant}: {img_path} [{deposit_type}]")

In [ ]:
# Insert E-Nose data into DB
print("Inserting E-Nose data into DB...")
aroma_count = 0
for idx, row in enose_df.iterrows():
    batch_id = generate_batch_id()
    pred_class, confidence = mock_predict_class(row["adulterant"])

    try:
        insert_aroma_analysis(
            batch_id=batch_id,
            ammonia=float(row["ammonia"]),
            alcohol=float(row["alcohol"]),
            voc=float(row["voc"]),
            sulfur=float(row["sulfur"]),
            hydrocarbon=float(row["hydrocarbon"]),
            predicted_class=pred_class,
            confidence=confidence,
            db_path=DB_PATH
        )
        aroma_count += 1
    except Exception as e:
        print(f"[ERROR] Row {idx}: {e}")
        break

    if (idx + 1) % 1000 == 0:
        print(f"  ...inserted {idx + 1}/{len(enose_df)} E-Nose samples")

print(f"[OK] Inserted {aroma_count} E-Nose samples into DB")

In [ ]:
# Insert E-Tongue data into DB
print("Inserting E-Tongue data into DB...")
taste_count = 0
for idx, row in etongue_df.iterrows():
    batch_id = generate_batch_id()
    pred_class, confidence = mock_predict_class(row["adulterant"])

    try:
        insert_taste_analysis(
            batch_id=batch_id,
            sweetness=float(row["sweetness"]),
            saltiness=float(row["saltiness"]),
            sourness=float(row["sourness"]),
            bitterness=float(row["bitterness"]),
            umami=float(row["umami"]),
            astringency=float(row["astringency"]),
            predicted_class=pred_class,
            confidence=confidence,
            db_path=DB_PATH
        )
        taste_count += 1
    except Exception as e:
        print(f"[ERROR] Row {idx}: {e}")
        break

    if (idx + 1) % 1000 == 0:
        print(f"  ...inserted {idx + 1}/{len(etongue_df)} E-Tongue samples")

print(f"[OK] Inserted {taste_count} E-Tongue samples into DB")

In [ ]:
# Insert Vision data into DB
print("Inserting Vision data into DB...")
vision_count = 0
for img_path, deposit_type, adulterant in vision_results:
    batch_id = generate_batch_id()
    pred_class, confidence = mock_predict_class(adulterant)

    try:
        insert_vision_analysis(
            batch_id=batch_id,
            image_path=img_path,
            deposit_type=deposit_type,
            predicted_class=pred_class,
            confidence=confidence,
            db_path=DB_PATH
        )
        vision_count += 1
    except Exception as e:
        print(f"[ERROR] {img_path}: {e}")
        break

    if (vision_count) % 100 == 0:
        print(f"  ...inserted {vision_count}/{len(vision_results)} Vision samples")

print(f"[OK] Inserted {vision_count} Vision samples into DB")

## 6. Create Demo Sample Mappings

Create 20 representative samples per class that will be used in the Gradio demo.

In [ ]:
from foodguard_lib import execute_query

# For each adulterant, select 20 representative samples
demo_mapping = {}
for adulterant in ADULTERANTS:
    # Get samples where prediction matches true class (high confidence)
    query = """SELECT * FROM aroma_analysis
               WHERE batch_id LIKE 'BATCH-%'
               AND predicted_class = ?
               LIMIT 20"""
    rows = execute_query(DB_PATH, query, (adulterant,))

    if rows:
        # Create a demo sample identifier
        sample_ids = [f"{adulterant[:3].upper()}-{i+1:03d}" for i in range(len(rows))]
        demo_mapping[adulterant] = {
            "sample_ids": sample_ids,
            "aroma_batch_ids": [dict(row)["batch_id"] for row in rows],
            "count": len(sample_ids)
        }

print("Demo sample mapping:")
for adulterant, mapping in demo_mapping.items():
    print(f"  {adulterant}: {mapping['count']} samples")

# Save demo mapping
import json
demo_mapping_json = json.dumps(demo_mapping, indent=2)
with open("../data/demo_samples_mapping.json", "w") as f:
    f.write(demo_mapping_json)

print("\n[OK] Demo sample mapping saved to data/demo_samples_mapping.json")

## 7. Verify Data in DB

In [ ]:
# Check aroma_analysis table
aroma_query = "SELECT COUNT(*) as count FROM aroma_analysis"
aroma_rows = execute_query(DB_PATH, aroma_query)
print(f"Total aroma_analysis records: {dict(aroma_rows[0])['count']}")

# Check taste_analysis table
taste_query = "SELECT COUNT(*) as count FROM taste_analysis"
taste_rows = execute_query(DB_PATH, taste_query)
print(f"Total taste_analysis records: {dict(taste_rows[0])['count']}")

# Check vision_analysis table
vision_query = "SELECT COUNT(*) as count FROM vision_analysis"
vision_rows = execute_query(DB_PATH, vision_query)
print(f"Total vision_analysis records: {dict(vision_rows[0])['count']}")

## 8. Sample Data Inspection

In [ ]:
# Show sample aroma record
query = "SELECT * FROM aroma_analysis LIMIT 1"
rows = execute_query(DB_PATH, query)
if rows:
    print("Sample Aroma Record:")
    for key, value in dict(rows[0]).items():
        print(f"  {key}: {value}")

print("\n" + "="*50 + "\n")

# Show sample taste record
query = "SELECT * FROM taste_analysis LIMIT 1"
rows = execute_query(DB_PATH, query)
if rows:
    print("Sample Taste Record:")
    for key, value in dict(rows[0]).items():
        print(f"  {key}: {value}")

print("\n" + "="*50 + "\n")

# Show sample vision record
query = "SELECT * FROM vision_analysis LIMIT 1"
rows = execute_query(DB_PATH, query)
if rows:
    print("Sample Vision Record:")
    for key, value in dict(rows[0]).items():
        print(f"  {key}: {value}")

## ✅ Data Generation Complete!

**Summary**:
- ✓ Generated ~7000 E-Nose samples (1000/class)
- ✓ Generated ~7000 E-Tongue samples (1000/class)
- ✓ Generated ~420 Vision deposit images (60/class)
- ✓ All data inserted into SQLite DB
- ✓ Mock ML predictions created and stored
- ✓ Demo sample mapping prepared

**Next Steps**:
1. Run `05_correlation_engine.ipynb` to test correlation logic
2. Run `06_food_safety_reasoning.ipynb` to test LLM integration
3. Run `08_gradio_demo.ipynb` for interactive demo